# Employee Data analysis
## A step by step guide on cleaning data for further analysis

# Part 1: Data Cleaning

In [2]:
import pandas as pd
import numpy as np

## Loading the data and making a copy of it to clean

In [3]:
dataframe = pd.read_csv('Messy_employee_dataset.csv')



In [4]:
df = dataframe.copy()

## Inspect the Data

In [5]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   str    
 1   First_Name         1020 non-null   str    
 2   Last_Name          1020 non-null   str    
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   str    
 5   Status             1020 non-null   str    
 6   Join_Date          1020 non-null   str    
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   str    
 9   Phone              1020 non-null   int64  
 10  Performance_Score  1020 non-null   str    
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), int64(1), str(8)
memory usage: 165.7 KB


In [7]:
df.isnull().sum()

Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

### After inspection of the data i found out that there are:
- No duplicates
- Missing Values in the Age and Salary column
- A splitable column
- Inconsistent Date formats

### This is where the main data cleaning comes in

## Part A: Numerical columns

### Starting with the age column, outliers are found before filling the missing values 

In [9]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [10]:
df['Age'].isnull().sum()

np.int64(211)

In [11]:
df['Age'].dtype

dtype('float64')

### Creating a function to find outliers


In [12]:
def find_outlier(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    higher_limit = Q3 + 1.5 * IQR

    return lower_limit, higher_limit

In [13]:
ageoutlier = find_outlier(df, 'Age')
print(ageoutlier)

(np.float64(2.5), np.float64(62.5))


In [14]:
agelower_limit = 2.5
agehigher_limit = 62.5

for index, row in df.iterrows():
    if (row['Age'] < agelower_limit or row['Age'] > agehigher_limit):
        print(row)

In [15]:
agemean = df['Age'].mean()
agemedian = df['Age'].median()
print(agemean)
print(agemedian)

32.48454882571075
30.0


In [16]:
df['Age'] = df['Age'].fillna(agemedian)

## Numerical Column: Salary

In [17]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,30.0,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,30.0,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [18]:
df['Salary'].isnull().sum()

np.int64(24)

In [19]:
df['Salary'].dtype

dtype('float64')

In [20]:
salaryoutlier = find_outlier(df, 'Salary')
print(salaryoutlier)

(np.float64(19520.177499999976), np.float64(149846.33750000002))


In [21]:
slower_limit = 19520.177499999976
shigher_limit = 149846.33750000002

for index, row in df.iterrows():
    if (row['Salary'] < slower_limit or row['Salary'] > shigher_limit):
        print(row)

In [22]:
salarymean = df['Salary'].mean()
salarymedian = df['Salary'].median()
print(salarymean)
print(salarymedian)

85155.05639558233
85547.87


In [23]:
df['Salary'] = df['Salary'].fillna(salarymedian)

In [24]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,30.0,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,30.0,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


## Part B Cleaning Categorical columns
### Categorical column: Department-Region

In [25]:
df[['Department', 'Region']] = df['Department_Region'].str.split('-', n = 1, expand = True)

In [26]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work,Department,Region
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True,DevOps,California
1,EMP1001,Bob,Brown,30.0,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True,Finance,Texas
2,EMP1002,Alice,Jones,30.0,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True,Admin,Nevada
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True,Admin,Nevada
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False,Cloud Tech,Florida


In [27]:
df = df.drop(columns = ['Department_Region'])

In [28]:
col = df.pop('Department')
df.insert(4, 'Department', col)

In [29]:
col = df.pop('Region')
df.insert(5, 'Region', col)

In [30]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department,Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps,California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,30.0,Finance,Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,30.0,Admin,Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin,Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech,Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


### Categorical column: Join_Date

In [31]:
df['Join_Date'].isnull().sum()

np.int64(0)

In [32]:
df['Join_Date'].dtype

<StringDtype(na_value=nan)>

In [33]:
df['Join_date'] = pd.to_datetime(df['Join_Date'], errors = 'coerce')

In [34]:
df['Join_Date']

0         4/2/2021
1        7/10/2020
2        12/7/2023
3       11/27/2021
4         1/5/2022
           ...    
1015     8/19/2023
1016     11/7/2021
1017     10/4/2023
1018    12/16/2024
1019     2/22/2021
Name: Join_Date, Length: 1020, dtype: str

In [35]:
df['Join_Date'] = pd.to_datetime(df['Join_Date'], format = 'mixed')

In [36]:
df['Join_Date'] = df['Join_Date'].dt.strftime('%d/%m/%Y')

In [37]:
df['Join_Date']

0       02/04/2021
1       10/07/2020
2       07/12/2023
3       27/11/2021
4       05/01/2022
           ...    
1015    19/08/2023
1016    07/11/2021
1017    04/10/2023
1018    16/12/2024
1019    22/02/2021
Name: Join_Date, Length: 1020, dtype: str

In [38]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department,Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work,Join_date
0,EMP1000,Bob,Davis,25.0,DevOps,California,Active,02/04/2021,59767.65,bob.davis@example.com,-1651623197,Average,True,2021-04-02
1,EMP1001,Bob,Brown,30.0,Finance,Texas,Active,10/07/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True,2020-07-10
2,EMP1002,Alice,Jones,30.0,Admin,Nevada,Pending,07/12/2023,88145.90,alice.jones@example.com,-5596363211,Good,True,2023-12-07
3,EMP1003,Eva,Davis,25.0,Admin,Nevada,Inactive,27/11/2021,69450.99,eva.davis@example.com,-3476490784,Good,True,2021-11-27
4,EMP1004,Frank,Williams,25.0,Cloud Tech,Florida,Active,05/01/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False,2022-01-05


In [39]:
df = df.drop(columns = 'Join_date')

In [40]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department,Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps,California,Active,02/04/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,30.0,Finance,Texas,Active,10/07/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,30.0,Admin,Nevada,Pending,07/12/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin,Nevada,Inactive,27/11/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech,Florida,Active,05/01/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


## Save the cleaned datasheet

In [41]:
df.to_csv('Cleaned employee dataset.csv', index = False)

## Grouping and Aggregation

In [42]:
df.groupby('Department')['Salary'].mean()

Department
Admin         85202.644096
Cloud Tech    84944.434658
DevOps        86003.638624
Finance       82279.519353
HR            85475.379474
Sales         86873.948258
Name: Salary, dtype: float64

In [43]:
df.groupby('Department')['Salary'].agg(['mean', 'sum','min', 'max'])

,mean,sum,min,max
Department,,,,
Admin,85202.644096,14143638.92,50110.66,119574.27
Cloud Tech,84944.434658,12401887.46,50047.32,119971.65
DevOps,86003.638624,16254687.70,50568.35,119586.11
Finance,82279.519353,13987518.29,50060.73,118672.76
HR,85475.379474,14616289.89,50557.84,119890.35
Sales,86873.948258,15463562.79,50288.67,119801.30


In [44]:
df.groupby('Region')['Salary'].mean()

Region
California    86620.397380
Florida       86293.965946
Illinois      85279.160182
Nevada        83640.458047
New York      83795.913292
Texas         85017.948758
Name: Salary, dtype: float64

In [46]:
df.groupby(['Department', 'Region'])['Salary'].mean()

Department  Region    
Admin       California    89564.108824
            Florida       86076.526800
            Illinois      92050.898621
            Nevada        80641.359412
            New York      78822.267727
            Texas         81871.540000
Cloud Tech  California    83877.977931
            Florida       91715.487857
            Illinois      85086.828750
            Nevada        85820.091000
            New York      78086.093333
            Texas         84466.728276
DevOps      California    82772.097143
            Florida       87116.325294
            Illinois      85678.447273
            Nevada        90001.822593
            New York      84959.466667
            Texas         86466.995185
Finance     California    87419.617188
            Florida       83098.247917
            Illinois      75707.702424
            Nevada        76398.523448
            New York      89159.340435
            Texas         83832.979310
HR          California    87745.022692
  

In [47]:
df.groupby(['Region', 'Department'])['Salary'].mean()

Region      Department
California  Admin         89564.108824
            Cloud Tech    83877.977931
            DevOps        82772.097143
            Finance       87419.617188
            HR            87745.022692
            Sales         88533.919677
Florida     Admin         86076.526800
            Cloud Tech    91715.487857
            DevOps        87116.325294
            Finance       83098.247917
            HR            80460.818293
            Sales         90582.736667
Illinois    Admin         92050.898621
            Cloud Tech    85086.828750
            DevOps        85678.447273
            Finance       75707.702424
            HR            87707.819167
            Sales         86982.183667
Nevada      Admin         80641.359412
            Cloud Tech    85820.091000
            DevOps        90001.822593
            Finance       76398.523448
            HR            85849.801250
            Sales         84886.507143
New York    Admin         78822.267727
  

### This means for every department in the region calculate the average salary

In [48]:
df.groupby('Department').agg({
    'Salary' : ['mean', 'max'],
    'Employee_ID' : 'count'
})

Salary            Employee_ID
                    mean        max       count
Department                                     
Admin       85202.644096  119574.27         166
Cloud Tech  84944.434658  119971.65         146
DevOps      86003.638624  119586.11         189
Finance     82279.519353  118672.76         170
HR          85475.379474  119890.35         171
Sales       86873.948258  119801.30         178

In [49]:
df.groupby(['Department', 'Region']).agg({
    'Salary' : ['mean', 'max'],
    'Employee_ID' : 'count'
})

Salary            Employee_ID
                               mean        max       count
Department Region                                         
Admin      California  89564.108824  119311.14          34
           Florida     86076.526800  119152.47          25
           Illinois    92050.898621  119574.27          29
           Nevada      80641.359412  115565.82          34
           New York    78822.267727  115311.68          22
           Texas       81871.540000  108661.01          22
Cloud Tech California  83877.977931  115959.65          29
           Florida     91715.487857  119389.15          28
           Illinois    85086.828750  114357.10          16
           Nevada      85820.091000  114963.34          20
           New York    78086.093333  114465.04          24
           Texas       84466.728276  119971.65          29
DevOps     California  82772.097143  115290.79          35
           Florida     87116.325294  115536.66          34
           Illinois    85678.447273  119586.11          33
           Nevada      90001.822593  117537.71          27
           New York    84959.466667  118288.91          33
           Texas       86466.995185  117870.22          27
Finance    California  87419.617188  117910.15          32
           Florida     83098.247917  118646.95          24
           Illinois    75707.702424  111742.40          33
           Nevada      76398.523448  118413.05          29
           New York    89159.340435  114822.78          23
           Texas       83832.979310  118672.76          29
HR         California  87745.022692  117267.63          26
           Florida     80460.818293  116509.72          41
           Illinois    87707.819167  118959.28          24
           Nevada      85849.801250  119890.35          24
           New York    86256.830303  118677.67          33
           Texas       88007.280870  118415.80          23
Sales      California  88533.919677  119801.30          31
           Florida     90582.736667  119407.93          33
           Illinois    86982.183667  119764.20          30
           Nevada      84886.507143  116695.26          35
           New York    83930.126154  118172.42          26
           Texas       85526.279565  110391.17          23

## Correlation

### Correlation measures the relationship between two numerical variables. It tells you whether they tend to increase or decrease together.

In [50]:
df['Age'].corr(df['Salary'])

np.float64(0.07676064389189492)

### Hence these two columns have no correlation at all

### Or to calculate correlation for all numeric columns

In [53]:
df.corr(numeric_only = True)

,Age,Salary,Phone,Remote_Work
Age,1.000000,0.076761,0.036646,0.016837
Salary,0.076761,1.000000,-0.057406,-0.033978
Phone,0.036646,-0.057406,1.000000,0.022841
Remote_Work,0.016837,-0.033978,0.022841,1.000000


### Hence there are no columns in this dataset that are correlated